# 03 · Vehicle mix: exposure vs harm

Crash involvement by vehicle group over time, contrasted with traffic volume by class from BHTrans radars (when a processed radar month is available — see notebook 04).

**Caveat:** radar classification (car / motorcycle / truck-bus) may undercount motorcycles riding between lanes; treat shares as indicative.

In [1]:
# Setup: works on Colab (Drive + private repo) and locally (repo checkout).
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    os.environ["BHTRAFFIC_DATA_ROOT"] = "/content/drive/MyDrive/transito_bh"
    token = userdata.get("GH_TOKEN")  # Colab Secrets: fine-grained token, read-only on the repo
    # Clone (not pip-from-git): the package resolves data/ and paper/ relative to the repo root.
    url = f"https://{token}@github.com/Dias0202/Transito_bh.git"
    if os.path.isdir("/content/Transito_bh"):
        !git -C /content/Transito_bh pull -q {url} main
    else:
        !git clone -q {url} /content/Transito_bh
        !git -C /content/Transito_bh remote set-url origin https://github.com/Dias0202/Transito_bh.git
    %pip -q install -e "/content/Transito_bh[geo,viz]"
    sys.path.insert(0, "/content/Transito_bh/src")
    os.chdir("/content/Transito_bh/notebooks")
else:
    from pathlib import Path
    sys.path.insert(0, str(Path.cwd().parent / "src"))

In [2]:
import pandas as pd
from bhtraffic.config import processed_dir
d = processed_dir('sinistros')
o = pd.read_parquet(d / 'ocorrencias.parquet')
v = pd.read_parquet(d / 'veiculos.parquet')
mix = v.merge(o[['boletim', 'year']], on='boletim').groupby(['year', 'grupo']).size().unstack(fill_value=0)
(mix.div(mix.sum(axis=1), axis=0) * 100).round(1)

grupo,bicycle,bus,car,motorcycle,other,truck
year,,,,,,
2011,1.4,8.0,48.8,36.7,2.7,2.4
2012,1.6,8.2,48.9,37.1,1.7,2.5
2013,1.5,8.7,49.4,36.2,1.9,2.3
2014,1.5,8.7,49.6,36.7,1.2,2.4
2015,1.6,7.6,49.4,37.5,1.8,2.2
2016,1.5,6.5,51.2,37.6,1.0,2.2
2017,1.5,5.5,50.9,38.9,1.1,2.1
2018,1.7,4.9,50.2,40.1,1.0,2.1
2019,1.9,4.5,50.0,40.6,0.9,2.0


In [3]:
radar_files = sorted(p for p in processed_dir('radares').glob('*.parquet') if not p.stem.endswith('_radars'))
[p.name for p in radar_files]

['AGOSTO_2021.parquet']

In [4]:
if radar_files:
    agg = pd.concat(pd.read_parquet(p) for p in radar_files)
    vol = agg.groupby('classe').n.sum()
    print('days covered:', agg.bin.dt.date.nunique())
    display((vol / vol.sum() * 100).round(1).rename('share of detected vehicles (%)'))
else:
    print('No radar month processed yet: run notebook 04 on Colab.')

days covered: 10


classe
car           82.0
motorcycle    12.5
other          0.0
truck_bus      5.5
Name: share of detected vehicles (%), dtype: float64

Frota (Senatran) is not on the PBH portal — see `docs/roadmap.md` for the manual download.